# Course 2 · Week 1 — Hands-on: Neural Network Forward Propagation

In Course 1 you fit one or two parameters at a time. Now you'll build a tiny **neural network** with 13 parameters — and watch it carve out a *circular* decision boundary that linear regression could never produce.

By the end you'll have:

1. Implemented the sigmoid activation
2. Implemented one fully-connected (dense) layer in numpy
3. Glued layers into a multi-layer forward pass
4. Used pre-trained weights from sklearn to classify points as inside/outside a circle
5. Drawn the learned decision boundary
6. (Stretch) Run the same code on a 3-layer network — your `forward()` handles arbitrary depth for free

Estimated time: **45–60 minutes.**


## Setup — points inside or outside a circle

This is the **simplest non-linear** classification problem you can imagine. Points inside the unit circle (radius 1) are class 1; points outside are class 0. A linear classifier can't solve this — no straight line separates them. We need a curved boundary, which means we need a non-linear model. Welcome to neural networks.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
import warnings; warnings.filterwarnings("ignore")

np.random.seed(0)
m = 200

# Generate points inside a 3×3 box; label = 1 if inside the unit circle, else 0.
X = np.random.uniform(-1.5, 1.5, (m, 2))
y = (X[:, 0]**2 + X[:, 1]**2 < 1.0).astype(float)
print(f"X shape: {X.shape}")
print(f"Points inside circle (class 1): {int(y.sum())}/{m}")
print(f"Points outside circle (class 0): {int((1-y).sum())}/{m}")


You should see green dots clustered inside a circle and red dots outside. The dashed black ring is the true boundary you want your model to recover.


In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(X[y==1, 0], X[y==1, 1], color="#10b981", s=40, label="inside (class 1)", alpha=0.7)
plt.scatter(X[y==0, 0], X[y==0, 1], color="#ef4444", s=40, label="outside (class 0)", alpha=0.7)
theta = np.linspace(0, 2*np.pi, 100)
plt.plot(np.cos(theta), np.sin(theta), color="black", lw=1.5, ls="--", label="true boundary")
plt.xlabel("$x_1$"); plt.ylabel("$x_2$"); plt.legend(); plt.grid(alpha=0.3); plt.axis("equal")
plt.title("The classification problem: inside or outside the unit circle?")
plt.show()


## Get pre-trained weights

We're not going to train the network ourselves this week — that requires backpropagation, which we'll cover next week. Instead, sklearn will train it for us, and we'll grab the weights it learned. Then *you'll* implement the forward pass that uses those weights.

After this cell runs, you'll have four arrays of numbers (`W1`, `b1`, `W2`, `b2`) — the trained parameters of a 2 → 4 → 1 neural network. Your job is to push the data through them.


In [ ]:
# Train a small neural net (2 → 4 → 1) using sklearn. We don't care about
# the training process here — we just want a set of trained weights.
clf = MLPClassifier(hidden_layer_sizes=(4,), activation="logistic",
                    max_iter=4000, random_state=42, tol=1e-6,
                    solver="adam", learning_rate_init=0.05)
clf.fit(X, y)
print(f"sklearn training accuracy: {clf.score(X, y):.4f}")

# Extract the weights — these are the numbers you'll plug into your own forward pass.
W1 = clf.coefs_[0]      # shape (2, 4)
b1 = clf.intercepts_[0]  # shape (4,)
W2 = clf.coefs_[1]      # shape (4, 1)
b2 = clf.intercepts_[1]  # shape (1,)

print(f"\nLayer 1: W1 shape {W1.shape}, b1 shape {b1.shape}")
print(f"Layer 2: W2 shape {W2.shape}, b2 shape {b2.shape}")


## Quick recap

A **neural network** is a stack of layers. Each layer takes a vector of numbers in, multiplies it by a weight matrix, adds a bias vector, and applies a non-linear function (the activation). The output of one layer becomes the input to the next.

For our 2 → 4 → 1 network:
- **Input:** 2 numbers (the point's `x_1, x_2` coordinates)
- **Hidden layer:** 4 neurons, each computing `sigmoid(w·x + b)`. Outputs a 4-dim vector.
- **Output layer:** 1 neuron, taking the 4-dim hidden vector and producing one number — the probability of "inside the circle".

The math:

$$\vec a^{[1]} = \sigma(\vec x^T W^{[1]} + \vec b^{[1]}), \quad \hat y = \sigma(\vec a^{[1] T} W^{[2]} + b^{[2]})$$

That's the whole network. Two matrix multiplies, two sigmoids. Done.


## Exercise 1 — sigmoid (again)

Same function as Course 1 Week 3. Implement `sigmoid(z) = 1 / (1 + e^{-z})`. Should work on scalars and on numpy arrays.


In [ ]:
def sigmoid(z):
    """Squash any real number (or array) into (0, 1)."""
    # TODO: implement using np.exp
    return 0.5  # placeholder

print(f"sigmoid(0)  = {sigmoid(0.0):.4f}   expected 0.5000")
print(f"sigmoid(2)  = {sigmoid(2.0):.4f}   expected 0.8808")
print(f"sigmoid(-2) = {sigmoid(-2.0):.4f}   expected 0.1192")
assert abs(sigmoid(0.0) - 0.5) < 1e-9
assert abs(sigmoid(2.0) - 0.8807970779778823) < 1e-9
print("✓ sigmoid() works")


## Exercise 2 — dense layer

A "dense" or "fully-connected" layer is just:

$$a_{\text{out}} = \text{activation}(a_{\text{in}} \cdot W + \vec b)$$

Three pieces: matrix multiply (`a_in @ W`), add bias (`+ b`), apply activation. Numpy makes this one or two lines.

The shapes:
- `a_in` is `(m, n_in)` — m examples, each with n_in features
- `W` is `(n_in, n_out)` — `n_out` neurons in this layer, each with `n_in` weights
- `b` is `(n_out,)` — one bias per neuron
- Result is `(m, n_out)`


In [ ]:
def dense(a_in, W, b, activation=sigmoid):
    """One layer of a neural network.

    a_in:       input activations, shape (m, n_in)
    W:          weights, shape (n_in, n_out)
    b:          biases, shape (n_out,)
    activation: function to apply element-wise after the linear part

    Returns a_out, shape (m, n_out).
    """
    # TODO: compute z = a_in @ W + b   then activation(z)
    return np.zeros((len(a_in), W.shape[1]))


# Quick test on layer 1 of our trained net
a1 = dense(X, W1, b1, sigmoid)
print(f"a1 shape: {a1.shape}   expected (200, 4)")
print(f"a1 first row: {np.round(a1[0], 4)}")
assert a1.shape == (200, 4), "Output of layer 1 should be (m, 4)"
# At every entry, sigmoid output is in (0, 1)
assert (a1 > 0).all() and (a1 < 1).all(), "sigmoid outputs should be strictly between 0 and 1"
print("✓ dense() works")


## Exercise 3 — multi-layer forward pass

Stack `dense()` calls. Start with `a = X`, then for each layer `(W, b, activation)`, compute `a = dense(a, W, b, activation)` and pass to the next layer.

The beauty: this code doesn't care how many layers there are. Two layers, ten layers, two hundred — same loop.


In [ ]:
def forward(X, layers):
    """Push data through a stack of layers, left to right.

    X:      input data, shape (m, n_features)
    layers: list of (W, b, activation) tuples — one per layer

    Returns the final activations, shape (m, n_outputs_of_last_layer).
    """
    # TODO: start with a = X, then for each layer apply your dense() function
    return np.zeros((len(X), 1))


# Run the trained 2-layer net
probs = forward(X, [(W1, b1, sigmoid), (W2, b2, sigmoid)])
print(f"output shape: {probs.shape}   expected (200, 1)")

y_pred = (probs.squeeze() >= 0.5).astype(float)
acc = float((y_pred == y).mean())
print(f"forward-pass accuracy: {acc:.4f}   expected ≈ 1.0000")
assert acc > 0.95, "your forward-pass accuracy should be near 1.0"
print("✓ forward() works")

# Sanity probe: a point dead-center (inside the circle) should be class 1
center_prob = forward(np.array([[0.0, 0.0]]), [(W1, b1, sigmoid), (W2, b2, sigmoid)]).item()
print(f"\np(class 1) for [[0, 0]]:    {center_prob:.4f}   (should be near 1)")
edge_prob = forward(np.array([[1.4, 1.4]]), [(W1, b1, sigmoid), (W2, b2, sigmoid)]).item()
print(f"p(class 1) for [[1.4, 1.4]]: {edge_prob:.4f}   (should be near 0)")


## Visualize the decision boundary

Now the payoff. Sample a 100×100 grid of points across the input space, run them all through `forward`, and shade by predicted probability. Add the original training points on top.

You should see:
- A **circular** green region in the center (the network thinks "class 1, inside")
- A **red** ring around it (the network thinks "class 0, outside")
- The dashed line marks the *true* boundary
- Your network's boundary should hug the true circle very closely

This is something logistic regression (Course 1) literally **cannot do**. Logistic regression draws a straight line. A neural network — even a tiny 2-layer one — bends the line into whatever shape the data needs.


In [ ]:
# Visualize the decision boundary the network learned
xs1, xs2 = np.meshgrid(np.linspace(-1.5, 1.5, 100), np.linspace(-1.5, 1.5, 100))
grid = np.column_stack([xs1.ravel(), xs2.ravel()])
grid_probs = forward(grid, [(W1, b1, sigmoid), (W2, b2, sigmoid)]).reshape(xs1.shape)

plt.figure(figsize=(7, 6))
plt.contourf(xs1, xs2, grid_probs, levels=20, cmap="RdYlGn", alpha=0.6)
plt.colorbar(label="P(class 1)")
plt.scatter(X[y==1, 0], X[y==1, 1], color="#10b981", s=30, edgecolors="black", lw=0.5, label="inside")
plt.scatter(X[y==0, 0], X[y==0, 1], color="#ef4444", s=30, edgecolors="black", lw=0.5, label="outside")
theta = np.linspace(0, 2*np.pi, 100)
plt.plot(np.cos(theta), np.sin(theta), color="black", lw=2, ls="--", label="true boundary")
plt.xlabel("$x_1$"); plt.ylabel("$x_2$"); plt.legend(); plt.axis("equal")
plt.title(f"Learned decision boundary — accuracy {acc:.1%}")
plt.show()


## ⭐ Stretch — go deeper

Train a `MLPClassifier(hidden_layer_sizes=(8, 4))` instead. That's a 2 → 8 → 4 → 1 network — three layers of weights instead of two.

Extract the new weights and pass them through your `forward()` function. Compare predictions to `clf_deep.predict()`.

The point isn't that deeper does better here (this problem is too easy — even one layer of 4 units already gets 100%). The point is: **your forward function handles arbitrary depth without changing a line of code.** The same five lines of `forward()` would run a network with a hundred layers.


In [ ]:
# STRETCH: deeper network
# Train a 2 → 8 → 4 → 1 network instead. Does it do better, the same, or worse?
# (Hint: hidden_layer_sizes=(8, 4) — sklearn handles arbitrary depths.)

clf_deep = MLPClassifier(hidden_layer_sizes=(8, 4), activation="logistic",
                         max_iter=4000, random_state=42, tol=1e-6,
                         solver="adam", learning_rate_init=0.05)
clf_deep.fit(X, y)

# TODO: extract weights from clf_deep (it now has 3 layers: 2→8, 8→4, 4→1)
# Build the layer list and pass it to your forward() function.
# Compare predictions to clf_deep.predict — should match perfectly.


## Wrap-up

You just built forward propagation from scratch. The same three-line `forward()` function would run a state-of-the-art network with hundreds of layers — the only thing that changes is the list of weights.

What you didn't do this week: train the weights. We let sklearn do it. Next week (Course 2 Week 2) you'll do **backpropagation**, which is the algorithm that produces these weights from data.

🎉 Course 2 Week 1 done.
